# Module 04.07 — Canvas Workpads (`canvas-workpad`)

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.7 Canvas Workpads (`canvas-workpad`)

Canvas is Kibana's **presentation layer** — it produces pixel-perfect, brand-able reports
and infographics driven by live Elasticsearch data. A workpad is a collection of pages;
each page contains elements (text, images, metrics, charts) whose data is fetched via
**Canvas expressions** embedded directly in the element definition.

The `canvas-workpad` saved object is unique among Kibana types because it has **no
external `references` array**. All data-fetching logic — the ES|QL queries, SQL
expressions, or `esaggs` calls — is embedded inline inside `pages[].elements[].expression`.
This makes workpads fully self-contained: you can export a single workpad object and it
carries everything it needs to run on any cluster, as long as the target indices exist.

From a restore perspective this is both a blessing and a constraint. The blessing: you
can restore just the workpad without worrying about dependency ordering. The constraint:
if the underlying index schema changes between snapshot and restore (different field names,
for example), the embedded expressions will break silently because Kibana has no way to
validate the field references at restore time.

### Create in Kibana UI
1. Go to **Canvas**
2. Click **Create workpad**
3. Add a metric element, configure with an ES|QL expression
4. Save (Canvas auto-saves on changes)

In [ ]:
heading('4.7 Canvas Workpad')

workpads = find_saved_objects('canvas-workpad')
console.print(f'  Found {len(workpads)} canvas workpad(s)')
if workpads:
    w = workpads[0]
    attrs = w['attributes']
    console.print(f"  Name    : {attrs.get('name')}")
    console.print(f"  Width   : {attrs.get('width')}, Height: {attrs.get('height')}")
    pages = attrs.get('pages', [])
    console.print(f"  Pages   : {len(pages)}")
    info('Key fields:')
    info('  pages[]         — array of page objects, each with elements[]')
    info('  elements[].expression — Canvas expression (essql, esaggs, esdocs, etc.)')
    info('  No external references — workpad is entirely self-contained')
else:
    info('No canvas workpads found — create one in the Kibana UI and re-run this cell.')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link('/app/canvas', 'Canvas — view workpads')

In [ ]:
heading('4.7 Canvas Workpad — create via API')

existing_wp = next((o for o in find_saved_objects('canvas-workpad') if o['attributes'].get('name') == 'Training Workpad'), None)
if existing_wp:
    wp_id = existing_wp['id']
    info(f'Workpad already exists: id={wp_id}')
else:
    wp_resp = kibana_post('/api/saved_objects/canvas-workpad', {
        'attributes': {
            'name': 'Training Workpad',
            'width': 1080,
            'height': 720,
            'css': '',
            'variables': [],
            'pages': [{
                'id': 'page-1',
                'style': {'background': '#fff'},
                'transition': {},
                'groups': [],
                'elements': [{
                    'id': 'el-1',
                    'position': {'top': 100, 'left': 100, 'width': 500, 'height': 100, 'angle': 0, 'parent': None},
                    'expression': 'markdown "## Snapshot Training\nThis workpad is used for snapshot restore exercises."',
                    'filter': None,
                }],
            }],
        },
        'references': [],
    })
    wp_id = wp_resp['id']
    success(f'Canvas workpad created: id={wp_id}')

obj = kibana_get(f'/api/saved_objects/canvas-workpad/{wp_id}')
info('Key fields:')
info('  name    — display name shown in Canvas workpad list')
info('  pages   — array of pages, each containing elements with Canvas expressions')
info('  width/height — workpad dimensions in pixels')
kibana_link('/app/canvas', 'Canvas — verify workpad appears in list')

In [ ]:
# Ensure workpad exists before snapshotting (re-runnable)
if not any(o['id'] == wp_id for o in find_saved_objects('canvas-workpad')):
    warn('Workpad missing — recreating')
    wp_resp = kibana_post('/api/saved_objects/canvas-workpad', {
        'attributes': {
            'name': 'Training Workpad', 'width': 1080, 'height': 720,
            'css': '', 'variables': [],
            'pages': [{'id': 'page-1', 'style': {'background': '#fff'},
                'transition': {}, 'groups': [],
                'elements': [{'id': 'el-1',
                    'position': {'top': 100, 'left': 100, 'width': 500, 'height': 100, 'angle': 0, 'parent': None},
                    'expression': 'markdown "## Snapshot Training"', 'filter': None}]}],
        },
        'references': [],
    })
    wp_id = wp_resp['id']

snap_delete_restore_cycle('snap-canvas-test', 'Canvas Workpad')

kibana_delete(f'/api/saved_objects/canvas-workpad/{wp_id}')
warn(f'Accidentally deleted workpad {wp_id}')
assert not any(o['id'] == wp_id for o in find_saved_objects('canvas-workpad')), 'Should be gone'

restore_kibana_state(es, SNAP_REPO, 'snap-canvas-test')
time.sleep(3)

restored = find_saved_objects('canvas-workpad')
assert any(o['id'] == wp_id for o in restored), 'Workpad should be restored'
success(f'Canvas workpad {wp_id} successfully restored!')
kibana_link('/app/canvas', 'Verify restored workpad in Canvas')